# AEGIS-CODE — analise a seguranca do seu projeto

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/felpsbc/SecTool/blob/main/AEGIS.ipynb)

Este notebook roda o AEGIS (analisador estatico OWASP Top 10) **sem instalar nada no seu computador**.
O codigo analisado **nunca e executado** — o AEGIS so le a arvore sintatica (AST).

**Como usar:** execute as celulas em ordem (`Ambiente de execucao → Executar tudo`, ou `Ctrl+F9`).

| Passo | O que faz |
| --- | --- |
| 1. Preparar | instala o AEGIS (~1 min) |
| 2. Validar a ferramenta | roda o AEGIS no corpus rotulado e mostra precisao/recall — prova que a ferramenta funciona |
| 3. Analisar o seu projeto | link do GitHub, upload de `.zip` ou pasta local |
| 4. Relatorio | tabela de achados, recomendacoes e download do JSON |

Linguagens: Python (Flask/Django), JavaScript/TypeScript, Java, PHP, HTML e arquivos de configuracao (`.env`, YAML, JSON, INI, TOML, `.properties`).

## 1. Preparar o ambiente

Nao precisa configurar nada: basta executar a celula abaixo.

Opcional: para analisar um repositorio **privado seu** na etapa 3, crie um token do GitHub com permissao de
leitura (<https://github.com/settings/tokens>) e salve-o no Colab em **🔑 Secrets** (barra lateral esquerda)
com o nome `GITHUB_TOKEN`, liberando o acesso deste notebook.

In [ ]:
#@title Instalar o AEGIS
import os, subprocess, sys
from pathlib import Path

REPO_AEGIS = "https://github.com/felpsbc/SecTool.git"  #@param {type:"string"}

def _token():
    if os.environ.get("GITHUB_TOKEN"):
        return os.environ["GITHUB_TOKEN"]
    try:
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None

def _com_token(url):
    token = _token()
    if token and url.startswith("https://github.com/"):
        return url.replace("https://", f"https://x-access-token:{token}@", 1)
    return url

# Rodando de um clone local (Jupyter/VS Code)? Usa o proprio repositorio.
_aqui = Path.cwd()
_locais = [p for p in (_aqui, *_aqui.parents) if (p / "TOOL" / "owaspscan").is_dir()]
if _locais:
    RAIZ_AEGIS = _locais[0]
else:
    RAIZ_AEGIS = Path("/content/AEGIS") if Path("/content").is_dir() else _aqui / "AEGIS"
    if not RAIZ_AEGIS.exists():
        r = subprocess.run(["git", "clone", "-q", "--depth", "1", _com_token(REPO_AEGIS), str(RAIZ_AEGIS)],
                           capture_output=True, text=True)
        if r.returncode != 0:
            raise RuntimeError(
                "Nao consegui clonar o AEGIS. Confira a conexao com a internet e o endereco "
                "em REPO_AEGIS.\n" + r.stderr.replace(_token() or "\0", "***"))

TOOL = RAIZ_AEGIS / "TOOL"
# Extra [linguagens] = tree-sitter para JS/TS, Java, PHP e HTML.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{TOOL}[linguagens]"], check=True)
if str(TOOL) not in sys.path:
    sys.path.insert(0, str(TOOL))

from owaspscan.varredura import VERSAO, executar
from owaspscan.relatorio import formatar_texto, para_dict
print(f"AEGIS / owasp-scan {VERSAO} pronto — {TOOL}")

## 2. Validar a ferramenta

O repositorio traz um **corpus rotulado**: cada vulnerabilidade plantada tem um comentario `# achado: CWE-xx`
na linha exata. A celula abaixo roda o AEGIS nesse corpus e compara com o gabarito.

- **Precisao** = dos achados apontados, quantos sao reais (falso positivo derruba).
- **Recall** = das vulnerabilidades reais, quantas foram encontradas (falso negativo derruba).
- `FN conh.` = limitacoes conhecidas e declaradas da ferramenta.

In [ ]:
#@title Metricas no corpus rotulado
from owaspscan.metricas import avaliar, formatar_tabela

corpus = TOOL / "tests" / "amostras"
v = executar(corpus)
print(formatar_tabela(avaliar(corpus, v), v))

In [ ]:
#@title Prova rapida: codigo vulneravel vs. codigo corrigido
amostras = TOOL / "tests" / "amostras"
for nome in ("vulneravel", "seguro"):
    v = executar(amostras / nome)
    print(f"{nome:>10}: {len(v.achados):3d} achados em {v.arquivos_analisados} arquivos")
print("\nEsperado: muitos achados em 'vulneravel' e (quase) nenhum em 'seguro'.")

## 3. Analisar o seu projeto

Escolha **uma** origem no formulario e execute a celula:

- **github** — cole o link do repositorio (para repositorio privado, use o `GITHUB_TOKEN` descrito no passo 1).
- **zip** — abre um botao para enviar um `.zip` do projeto.
- **pasta_local** — caminho de uma pasta ou arquivo ja disponivel nesta maquina (Jupyter local, ou Google Drive montado no Colab).

In [ ]:
#@title Escolher o projeto
ORIGEM = "github"  #@param ["github", "zip", "pasta_local"]
LINK_GITHUB = "https://github.com/usuario/projeto"  #@param {type:"string"}
PASTA_LOCAL = ""  #@param {type:"string"}

import shutil, tempfile, zipfile

TRABALHO = Path(tempfile.gettempdir()) / "aegis_projeto"
shutil.rmtree(TRABALHO, ignore_errors=True)
TRABALHO.mkdir(parents=True)

def _extrair_zip(arquivo, destino):
    with zipfile.ZipFile(arquivo) as z:
        for membro in z.namelist():  # evita zip slip ("../" saindo do destino)
            if not (destino / membro).resolve().is_relative_to(destino.resolve()):
                raise ValueError(f"zip com caminho suspeito: {membro}")
        z.extractall(destino)

if ORIGEM == "github":
    PROJETO = TRABALHO / "repo"
    r = subprocess.run(["git", "clone", "-q", "--depth", "1", _com_token(LINK_GITHUB.strip()), str(PROJETO)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("Falha ao clonar: " + r.stderr.replace(_token() or "\0", "***"))
elif ORIGEM == "zip":
    PROJETO = TRABALHO / "zip"
    try:
        from google.colab import files
        for nome, dados in files.upload().items():
            (TRABALHO / nome).write_bytes(dados)
            _extrair_zip(TRABALHO / nome, PROJETO)
    except ImportError:
        raise RuntimeError("Upload so funciona no Colab. Fora dele, use ORIGEM = 'pasta_local'.")
else:
    PROJETO = Path(PASTA_LOCAL).expanduser()
    if not PASTA_LOCAL or not PROJETO.exists():
        raise FileNotFoundError(f"Caminho nao encontrado: {PASTA_LOCAL!r}")

print(f"Projeto pronto para analise: {PROJETO}")

## 4. Relatorio

In [ ]:
#@title Analisar
varredura = executar(PROJETO)
relatorio = para_dict(varredura)
r = relatorio["resumo"]
print(f"Arquivos analisados : {relatorio['varredura']['arquivos_analisados']} "
      f"(+ {relatorio['varredura']['arquivos_de_configuracao']} de configuracao, "
      f"{relatorio['varredura']['arquivos_com_falha']} com falha de leitura)")
print(f"Achados             : {r['total']} "
      f"({r['rastreados_por_taint']} rastreados por taint, {r['candidatos_heuristicos']} candidatos heuristicos)")
print(f"Por severidade      : {r['por_severidade']}")
print("\nVeredito:", "⚠️  ha achados de severidade ALTA — corrija antes do deploy."
      if varredura.tem_severidade_alta else "✅ nenhum achado de severidade alta.")
print("Candidatos heuristicos (confianca baixa) pedem revisao manual — nao sao certeza.")

try:
    import pandas as pd
    from IPython.display import display
    if relatorio["achados"]:
        ordem = {"alta": 0, "media": 1, "baixa": 2}
        tabela = (pd.DataFrame(relatorio["achados"])
                  [["severidade", "confianca", "cwe", "vulnerabilidade", "arquivo", "linha", "trecho", "owasp"]]
                  .sort_values(["severidade", "arquivo", "linha"], key=lambda c: c.map(ordem) if c.name == "severidade" else c))
        display(tabela.reset_index(drop=True))
except ImportError:
    pass

In [ ]:
#@title Como corrigir cada achado (explicacao e recomendacao)
from owaspscan.relatorio import ordenar
if not varredura.achados:
    print("Nenhum achado.")
for i, a in enumerate(ordenar(varredura.achados), 1):
    print(f"{i}. [{a.severidade.upper()} / confianca {a.confianca}] {a.cwe} {a.vulnerabilidade}")
    print(f"   {a.arquivo}:{a.linha}  ({a.owasp})")
    print(f"   > {a.trecho}")
    print(f"   Problema : {a.descricao}")
    print(f"   Correcao : {a.recomendacao}\n")

In [ ]:
#@title Baixar relatorio JSON
import json
saida = Path(tempfile.gettempdir()) / "relatorio_aegis.json"
saida.write_text(json.dumps(relatorio, ensure_ascii=False, indent=2))
try:
    from google.colab import files
    files.download(str(saida))
except ImportError:
    print(f"Relatorio salvo em {saida}")

---
Encontrou um falso positivo ou uma vulnerabilidade que o AEGIS deixou passar? Abra uma issue no repositorio
com o trecho de codigo — isso alimenta diretamente o corpus de avaliacao.